# Lecture 5.9 — Output Guardrails: Validating Agent Responses Before Delivery

In this notebook you'll build output guardrails: checks that run on an agent's final output right before that output is handed back to the caller. You'll see two structural differences that set output guardrails apart from input guardrails, and build working examples of each.

## Cell 1: Install the OpenAI Agents SDK

This notebook builds agents with output guardrails, using the OpenAI Agents SDK. The cell below installs the package.

If `openai-agents` is already installed in this Colab session, pip detects that it's satisfied and the install finishes almost instantly. There's no harm in re-running this cell if you're not sure whether it's been installed yet.

We pin the version below so every example in this notebook behaves the way it's described here. You can remove the pin to grab the latest release, or substitute a version of your own choosing.

In [1]:
# Pinned for reproducibility. To use the latest version,
# run: pip install openai-agents
# Or substitute your preferred version below.
!pip install openai-agents==0.18.3 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 880.8/880.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.4/223.4 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.4 MB/s eta 0:00:00


## Cell 2: Set Up Your OpenAI API Key

The SDK needs your OpenAI API key to make requests. In Colab, the safest way to provide it is through Colab's built-in Secrets manager, which keeps the key out of the notebook itself.

Steps to add your key:
1. Click the key icon in the left sidebar to open the **Secrets** panel.
2. Click **Add new secret**.
3. Set the name to `OPENAI_API_KEY`.
4. Paste your actual API key as the value.
5. Toggle **Notebook access** on so this notebook can read it.

The cell below reads that secret and sets it as an environment variable, which is how the SDK expects to find it.

**Running locally instead of Colab?** Skip the `userdata.get()` call. Set the key as an environment variable in your terminal before launching Jupyter, for example `export OPENAI_API_KEY="your-key-here"` on macOS or Linux, then the SDK will pick it up the same way.

In [2]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## Cell 3: Set the Model Name

Every agent in this notebook points at the same model, set once here as `MODEL_NAME`. Changing this single variable updates the model used everywhere an agent is defined below, so you never have to hunt through multiple cells to swap models.

See the comment in the cell for a link to OpenAI's current model list, since new models are added over time.

In [3]:
# See latest models at: https://platform.openai.com/docs/models
MODEL_NAME = "gpt-5.4-mini"

## Cell 4: Imports

This cell imports everything the notebook uses. A quick rundown:

| Import | Source | Purpose |
|---|---|---|
| `BaseModel` | `pydantic` | Defines structured output types, used here for a guardrail's own structured check result |
| `Reasoning` | `openai.types.shared` | Configures reasoning effort on agents that use reasoning models |
| `Agent` | `agents` | The core building block: an LLM configured with instructions, tools, and guardrails |
| `GuardrailFunctionOutput` | `agents` | The return type every guardrail function must produce |
| `ModelSettings` | `agents` | Configures per-agent model behaviour like reasoning effort and verbosity |
| `OutputGuardrailTripwireTriggered` | `agents` | The exception raised when an output guardrail's tripwire fires |
| `RunConfig` | `agents` | Run-level configuration, including guardrails that apply across a whole run |
| `RunContextWrapper` | `agents` | Wraps the run's context object, passed into every guardrail function |
| `Runner` | `agents` | Executes an agent (or chain of agents) and returns a result |
| `function_tool` | `agents` | Decorator that turns a plain Python function into a tool an agent can call |
| `output_guardrail` | `agents` | Decorator that turns a function into an output guardrail, the subject of this lecture |

`output_guardrail` is imported from the `agents` top-level package, the same place its counterpart for validating input lives. Both decorators follow the same overall shape, but as you'll see shortly, output guardrails have two structural differences worth understanding properly.

In [4]:
from pydantic import BaseModel

from openai.types.shared import Reasoning

from agents import (
    Agent,
    GuardrailFunctionOutput,
    ModelSettings,
    OutputGuardrailTripwireTriggered,
    RunConfig,
    RunContextWrapper,
    Runner,
    function_tool,
    output_guardrail,
)

## Cell 5: What Output Guardrails Do

Output guardrails are checks that run on the final output of an agent. Where an input guardrail looks at what the user typed before the agent gets to work, an output guardrail looks at what the agent produced, right before that answer is handed back.

If a guardrail's `tripwire_triggered` flag comes back `True`, the SDK raises `OutputGuardrailTripwireTriggered` and the response never reaches the caller.

That much sounds identical to input guardrails. But two structural differences set output guardrails apart, and they're the core of this lecture:

1. **Only the last agent's guardrails run.** In a run involving a handoff, the SDK checks the output guardrails belonging to whichever agent produced the final output, not the guardrails of every agent that took part along the way.
2. **There is no `run_in_parallel` option.** The dataclass field simply doesn't exist for output guardrails. You'll see exactly why once you look at the timing involved.

Both differences make sense once you think about what an output guardrail is actually checking. Keep that thought in mind as you work through the cells below.

## Cell 6: The @output_guardrail Decorator and Function Signature

Now let's build one. Every output guardrail function has this shape:

| Parameter | Type | What it receives |
|---|---|---|
| `ctx` | `RunContextWrapper` | The run's context object, same as any guardrail |
| `agent` | `Agent` | The agent whose output is being checked |
| `output` | `Any` | The agent's final output, whatever type that agent produces |

That third parameter is the detail to notice. An input guardrail's third parameter is the user's input. An output guardrail's third parameter is the agent's output. Same position in the signature, different direction of data flow.

The cell below defines a small policy-checking agent, then wraps a guardrail function around it with `@output_guardrail`. The guardrail runs a second agent internally to decide whether the checked text mentions a competitor by name, and returns that decision as a structured `GuardrailFunctionOutput`.

Notice the guardrail is attached to `sales_agent` through its `output_guardrails` list, the same pattern you'd use for input guardrails, just on the output side.

The decorator itself supports the same two calling forms you'd expect: bare `@output_guardrail`, or `@output_guardrail(name=...)`. What's missing is any `run_in_parallel` keyword. It isn't optional here. It doesn't exist.

In [5]:
class ContentPolicyOutput(BaseModel):
    is_compliant: bool
    reasoning: str


policy_checker_agent = Agent(
    name="Policy Checker",
    instructions=(
        "Check if the given text mentions competitor "
        "products by name. Flag it as non-compliant if "
        "it does."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    output_type=ContentPolicyOutput,
)


@output_guardrail
async def content_policy_guardrail(
    ctx: RunContextWrapper,
    agent: Agent,
    output: str,
) -> GuardrailFunctionOutput:
    result = await Runner.run(
        policy_checker_agent,
        f"Check this text: {output}",
        context=ctx.context,
    )
    check: ContentPolicyOutput = result.final_output
    return GuardrailFunctionOutput(
        output_info=check,
        tripwire_triggered=not check.is_compliant,
    )


sales_agent = Agent(
    name="Sales Agent",
    instructions="You are a helpful sales assistant.",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    output_guardrails=[content_policy_guardrail],
)

## Cell 7: Running It, Compliant and Non-Compliant Outputs

Time to see the guardrail actually check something. The cell below runs `sales_agent` twice.

The first run asks a plain question. Nothing about competitors comes up, so `content_policy_guardrail` should let the answer through untouched.

The second run deliberately invites `sales_agent` to compare itself against a named competitor, "BrandX." If the agent takes the bait, the guardrail should catch it and trip.

When the tripwire fires, `OutputGuardrailTripwireTriggered` carries a `guardrail_result` attribute pointing to an `OutputGuardrailResult`. Two of its fields are worth calling out specifically, because `InputGuardrailResult` does not have them: `agent`, the agent that was checked, and `agent_output`, the actual output that got flagged. Input guardrails only ever check one thing, the input, so they have no need for these. Output guardrails check something produced mid-run, so knowing which agent and which output tripped the wire matters.

Run the cell and read both outcomes.

In [6]:
try:
    result = await Runner.run(
        sales_agent,
        "What kind of laptops do you sell?",
    )
    print("Compliant result:", result.final_output)
except OutputGuardrailTripwireTriggered:
    print("Unexpectedly blocked")

try:
    result = await Runner.run(
        sales_agent,
        (
            "Tell me honestly, are we better than our competitior BrandX "
            "laptops? Feel free to mention them directly "
            "in your comparison."
        ),
    )
    print("Result:", result.final_output)
except OutputGuardrailTripwireTriggered as e:
    print("Output guardrail tripped!")
    print(
        f"Guardrail name: "
        f"{e.guardrail_result.guardrail.get_name()}"
    )
    info: ContentPolicyOutput = (
        e.guardrail_result.output.output_info
    )
    print(f"Reasoning: {info.reasoning}")
    print(
        f"Blocked output was for agent: "
        f"{e.guardrail_result.agent.name}"
    )
    print(
        f"Blocked output content: "
        f"{e.guardrail_result.agent_output}"
    )

Compliant result: We sell a range of laptops, including:

- **Everyday laptops** for browsing, email, and office work  
- **Business laptops** with strong security and reliability  
- **Gaming laptops** with powerful graphics and fast processors  
- **Ultrabooks** that are thin, light, and portable  
- **2-in-1 laptops** with touchscreen and tablet mode  
- **Budget laptops** for basic tasks and students  

If you want, I can also help you choose one based on your budget and use case.
Output guardrail tripped!
Guardrail name: content_policy_guardrail
Reasoning: The text mentions a competitor product by name: "BrandX laptops."
Blocked output was for agent: Sales Agent
Blocked output content: I can help compare you to **BrandX laptops**, but I can’t honestly say “you’re better” without knowing which model, specs, price, and what “better” means for your customer.

A fair comparison usually comes down to:
- **Performance**: CPU/GPU, RAM, thermals
- **Battery life**
- **Build quality**
- **

## Cell 8: Output Guardrails Only Run on the Last Agent

This cell demonstrates the first structural difference from Cell 5: only the last agent's output guardrails run, not every agent that took part in a run. To make that concrete, this cell runs two different questions through the same `front_desk` agent, one that triggers a handoff and one that doesn't.

`front_desk` has no output guardrails at all. `pricing_agent` does, the same `content_policy_guardrail` from before. `front_desk` is instructed to hand pricing questions off to `pricing_agent` and answer everything else itself.

**Scenario 1** asks about a laptop's price. `front_desk` hands off, control transfers, and `pricing_agent` becomes `result.last_agent`. Since `pricing_agent` carries `content_policy_guardrail`, that guardrail is the one that runs on the final output.

**Scenario 2** asks about store hours, which has nothing to do with pricing. `front_desk` answers directly, with no handoff at all. `front_desk` stays as `result.last_agent`, but `front_desk.output_guardrails` is an empty list, so nothing checks the output.

Watch the two printed "Output guardrails on last agent" lines. The first should show `content_policy_guardrail`. The second should show an empty list. Same guardrail, defined once, but whether it runs at all depends entirely on which agent ends up finishing the run.

In [7]:
@function_tool
def lookup_price(item: str) -> str:
    """Looks up the price of an item.

    Args:
        item: The item to look up.
    """
    return f"{item}: $299"


pricing_agent = Agent(
    name="Pricing Agent",
    instructions=(
        "You look up prices using the lookup_price tool."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[lookup_price],
    output_guardrails=[content_policy_guardrail],
)

front_desk = Agent(
    name="Front Desk",
    instructions=(
        "You are a front desk agent. "
        "Hand off pricing questions to the Pricing Agent. "
        "Answer any other question yourself directly."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    handoffs=[pricing_agent],
)

handoff_result = await Runner.run(
    front_desk,
    "How much is a laptop?",
)
print("Scenario 1: pricing question (triggers a handoff)")
print("Final output:", handoff_result.final_output)
print("Last agent:", handoff_result.last_agent.name)
print(
    "Output guardrails on last agent:",
    [g.get_name() for g in handoff_result.last_agent.output_guardrails],
)

print()

direct_result = await Runner.run(
    front_desk,
    "What are your store hours?",
)
print("Scenario 2: non-pricing question (front_desk answers directly)")
print("Final output:", direct_result.final_output)
print("Last agent:", direct_result.last_agent.name)
print(
    "Output guardrails on last agent:",
    [g.get_name() for g in direct_result.last_agent.output_guardrails],
)

Scenario 1: pricing question (triggers a handoff)
Final output: I can look that up, but I need a bit more detail—“laptop” can vary a lot by brand and model. If you want, I can price a specific one.
Last agent: Pricing Agent
Output guardrails on last agent: ['content_policy_guardrail']

Scenario 2: non-pricing question (front_desk answers directly)
Final output: We’re open daily from 9 AM to 9 PM.
Last agent: Front Desk
Output guardrails on last agent: []


## Cell 9: RunConfig.output_guardrails, Concatenated Not Overridden

`RunConfig` can also carry output guardrails, applied at the run level rather than attached to a specific agent. The question worth asking is what happens when an agent already has its own output guardrails and the run also specifies some: does one list win, or do both run?

For output guardrails, both run. The SDK builds its final guardrail list by combining the last agent's own `output_guardrails` with `RunConfig.output_guardrails`, and it's a straightforward list concatenation, not a replacement. Every guardrail from both sources checks the output.

To see that concretely rather than just take it on faith, this cell defines a second, deliberately different guardrail, `word_count_guardrail`. Unlike `content_policy_guardrail`, it doesn't call another agent at all. It's a plain synchronous function that counts words and never trips, which is a useful reminder that a guardrail function can be as simple or as involved as the check requires. `sales_agent` already carries `content_policy_guardrail` directly. This cell attaches `word_count_guardrail` separately, through `RunConfig`, and runs an ordinary return-policy question.

`RunResult` exposes an `output_guardrail_results` list, populated on every successful run, not just when a tripwire fires. Printing it after the run gives direct proof of concatenation: you'll see one entry for `content_policy_guardrail`, sourced from `sales_agent` itself, and a second entry for `word_count_guardrail`, sourced from `RunConfig`. Both ran, on the same output, in the same call.

In [8]:
@output_guardrail
def word_count_guardrail(
    ctx: RunContextWrapper,
    agent: Agent,
    output: str,
) -> GuardrailFunctionOutput:
    word_count = len(output.split())
    return GuardrailFunctionOutput(
        output_info={"word_count": word_count},
        tripwire_triggered=False,
    )


result = await Runner.run(
    sales_agent,
    "Tell me about your return policy.",
    run_config=RunConfig(
        workflow_name="Output guardrail demo",
        output_guardrails=[word_count_guardrail],
    ),
)
print("Result:", result.final_output)
print()
print("Output guardrails that ran on this result:")
for r in result.output_guardrail_results:
    print(f"  - {r.guardrail.get_name()} (output_info: {r.output.output_info})")

Result: I can help, but I don’t have a specific store or company policy to reference.

If you’re asking about our policy in general:
- Returns are typically accepted within a set window (often 14–30 days)
- Items usually must be unused and in original packaging
- Some items may be non-returnable, like final sale, perishables, or personalized products
- Refunds are often issued to the original payment method after inspection

If you want, send me the store/product name and I’ll help you find the exact return policy.

Output guardrails that ran on this result:
  - word_count_guardrail (output_info: {'word_count': 87})
  - content_policy_guardrail (output_info: is_compliant=True reasoning='The text does not mention any competitor products or brand names.')


## Cell 10: Input vs Output Guardrails, a Comparison

With both guardrail types now covered, here's the full picture side by side.

| | Input guardrails | Output guardrails |
|---|---|---|
| Checks | The user's input | The agent's final output |
| Runs on | The first agent only | The last agent only |
| `run_in_parallel` option | Yes, `True` by default or `False` | Does not exist |
| Timing | Concurrent with, or before, the agent | Always after the agent completes |
| Third function parameter | `input: str \| list` | `output: Any` |
| `RunConfig` field behaviour | Supplements the agent's own list | Concatenated with the agent's own list |
| Result type extras | — | `agent: Agent`, `agent_output: Any` |
| Exception raised | `InputGuardrailTripwireTriggered` | `OutputGuardrailTripwireTriggered` |

Most rows should already feel familiar from what you've just built. The two rows worth re-reading are "Runs on" and "run_in_parallel option," since those are the two genuinely structural differences this lecture set out to teach.

## Cell 11: Why No run_in_parallel Exists

It's worth spending a moment on why `run_in_parallel` doesn't exist for output guardrails, rather than just accepting it as a fact to memorize.

An input guardrail can run concurrently with the agent because both are working from the same starting point, the user's message. There's something for the guardrail to check the instant the run begins, so running it alongside the agent is a real option.

An output guardrail checks the agent's final output. That output doesn't exist until the agent has finished generating it. There is nothing to run "in parallel with," because the thing being checked hasn't been produced yet. This is a structural difference, not a missing configuration knob.

One nuance worth knowing: if an agent has multiple output guardrails attached, those guardrails do still run concurrently with each other, just not with the agent itself. The SDK schedules them as separate tasks and lets them race using `asyncio.as_completed`, so multiple checks on the same output complete in parallel. The one thing that's impossible is starting a check before the agent has something to check.